In [ ]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Split + Scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Feature Selection
from sklearn.feature_selection import SequentialFeatureSelector

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Tuning
from sklearn.model_selection import GridSearchCV

In [ ]:
df = pd.read_csv("../PHASE-1/Data/heart_cleaned.csv")
df.head()

In [ ]:
print(df.shape)

In [ ]:
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
lr = LogisticRegression(
    solver='liblinear',
    max_iter=5000
)

forward_selector = SequentialFeatureSelector(
    lr,
    n_features_to_select=5,
    direction='forward',
    cv=5
)

forward_selector.fit(
    X_train_scaled,
    y_train
)

forward_features = X.columns[
    forward_selector.get_support()
]

print("Forward Selected Features:")
print(forward_features)

Forward Selected Features

In [ ]:
X_train_forward = X_train[forward_features]
X_test_forward = X_test[forward_features]

In [ ]:
scaler_f = StandardScaler()

X_train_forward = scaler_f.fit_transform(X_train_forward)
X_test_forward = scaler_f.transform(X_test_forward)

In [ ]:
backward_selector = SequentialFeatureSelector(
    lr,
    n_features_to_select=5,
    direction='backward',
    cv=5
)

backward_selector.fit(
    X_train_scaled,
    y_train
)

backward_features = X.columns[
    backward_selector.get_support()
]

print("Backward Selected Features:")
print(backward_features)

Backward Selected Features

In [ ]:
X_train_backward = X_train[backward_features]
X_test_backward = X_test[backward_features]

In [ ]:
scaler_b = StandardScaler()

X_train_backward = scaler_b.fit_transform(X_train_backward)
X_test_backward = scaler_b.transform(X_test_backward)

In [ ]:
models = {
    "Logistic Regression":
        LogisticRegression(max_iter=5000),

    "KNN":
        KNeighborsClassifier(),

    "SVM":
        SVC(probability=True),

    "Decision Tree":
        DecisionTreeClassifier(),

    "Random Forest":
        RandomForestClassifier()
}

In [ ]:
def evaluate_models(Xtr, Xte, ytr, yte):

    results=[]

    for name, model in models.items():

        model.fit(Xtr,ytr)

        pred = model.predict(Xte)

        acc = accuracy_score(yte,pred)
        prec = precision_score(yte,pred)
        rec = recall_score(yte,pred)
        f1 = f1_score(yte,pred)

        roc = roc_auc_score(
            yte,
            model.predict_proba(Xte)[:,1]
        )

        results.append(
            [name,acc,prec,rec,f1,roc]
        )

    return pd.DataFrame(
        results,
        columns=[
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC_AUC"
        ]
    )

In [ ]:
full_results = evaluate_models(
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

full_results

In [ ]:
forward_results = evaluate_models(
    X_train_forward,
    X_test_forward,
    y_train,
    y_test
)

forward_results

In [ ]:
backward_results = evaluate_models(
    X_train_backward,
    X_test_backward,
    y_train,
    y_test
)

backward_results

In [ ]:
rf = RandomForestClassifier()

rf.fit(
    X_train_forward,
    y_train
)

pred = rf.predict(
    X_test_forward
)

cm = confusion_matrix(
    y_test,
    pred
)

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)
plt.show()

Best Classifier

In [ ]:
print("Full Features")
print(full_results)

print("Forward Selection")
print(forward_results)

print("Backward Selection")
print(backward_results)

In [ ]:
param_grid = {
    'n_estimators':[100,200,300],
    'max_depth':[3,5,10,None]
}

grid = GridSearchCV(
    RandomForestClassifier(),
    param_grid,
    cv=5
)

grid.fit(
    X_train_forward,
    y_train
)

print(grid.best_params_)

In [ ]:
best_rf = grid.best_estimator_

pred = best_rf.predict(
    X_test_forward
)

print(
accuracy_score(
y_test,
pred
)
)